# RoamAI — PySpark analytics pipeline

Reads the `activities` table from Lakebase via Spark JDBC, computes aggregate metrics (activity counts, average duration, weather sensitivity breakdown), and writes the results back as an `activity_analytics` table.

This is the "data pipeline in Spark" for the project. Uses standard PySpark DataFrame operations that would scale to millions of rows if the dataset grew.

## Prerequisites

* Lakebase URL stored in the `database/lakebase-url` secret (already set up)
* Activities table populated with the 15 Kauai seed rows

## 1. Load Lakebase connection details from the secret

In [0]:
import base64
from urllib.parse import urlparse

from databricks.sdk import WorkspaceClient

# Fetch and decode the Lakebase URL (stored as base64)
w = WorkspaceClient()
secret = w.secrets.get_secret(scope="database", key="lakebase-url")
LAKEBASE_URL = base64.b64decode(secret.value).decode("utf-8")

# Parse into components for Spark JDBC
parsed = urlparse(LAKEBASE_URL)
JDBC_URL = f"jdbc:postgresql://{parsed.hostname}:{parsed.port}{parsed.path}"
JDBC_USER = parsed.username
JDBC_PASS = parsed.password

print(f"JDBC URL: jdbc:postgresql://{parsed.hostname}:{parsed.port}{parsed.path}")
print(f"User:     {JDBC_USER}")
print(f"Password: {'*' * 12}")

JDBC URL: jdbc:postgresql://ep-royal-bread-d8tjlq4f-pooler.database.us-east-2.cloud.databricks.com:None/databricks_postgres
User:     student
Password: ************


## 2. Read the activities table via Databricks SDK 

In [0]:
# Build JDBC URL with default port if not specified
port = parsed.port or 5432
jdbc_url_with_port = f"jdbc:postgresql://{parsed.hostname}:{port}{parsed.path}"

activities_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url_with_port)
    .option("dbtable", "activities")
    .option("user", JDBC_USER)
    .option("password", JDBC_PASS)
    .option("driver", "org.postgresql.Driver")
    .load()
)

# Drop the vector column so we can display cleanly (Spark can't render vectors)
activities_df = activities_df.drop("description_embedding")

print(f"Loaded {activities_df.count()} activity rows")
activities_df.printSchema()

Loaded 15 activity rows
root
 |-- id: long (nullable = true)
 |-- destination_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- description: string (nullable = true)
 |-- weather_sensitive: boolean (nullable = true)
 |-- duration_hours: decimal(4,1) (nullable = true)
 |-- source: string (nullable = true)
 |-- created_at: timestamp (nullable = true)



## 3. Preview the raw data

In [0]:
activities_df.select(
    "id", "name", "category", "weather_sensitive", "duration_hours"
).show(20, truncate=False)

+---+-------------------------+------------+-----------------+--------------+
|id |name                     |category    |weather_sensitive|duration_hours|
+---+-------------------------+------------+-----------------+--------------+
|4  |Napali Coast Boat Tour   |water_sports|true             |5.0           |
|6  |Hanalei Bay              |beach       |true             |3.0           |
|7  |Sleeping Giant Trail     |hiking      |true             |2.5           |
|1  |Kalalau Trail            |hiking      |true             |8.0           |
|2  |Poipu Beach              |beach       |true             |3.0           |
|3  |Waimea Canyon            |viewpoint   |true             |2.5           |
|5  |Wailua River Kayak       |water_sports|true             |4.0           |
|11 |Wailua Falls             |viewpoint   |true             |0.5           |
|12 |Kokee State Park         |nature      |true             |4.0           |
|13 |Kauai Coffee Company Tour|cultural    |false            |1.

## 4. Aggregation: activities by category

Classic groupBy + aggregation. Counts and average duration per category, plus the count of weather-sensitive vs weather-safe activities.

In [0]:
from pyspark.sql import functions as F

by_category = (
    activities_df
    .groupBy("category")
    .agg(
        F.count("*").alias("activity_count"),
        F.round(F.avg("duration_hours"), 2).alias("avg_duration_hrs"),
        F.round(F.sum("duration_hours"), 2).alias("total_duration_hrs"),
        F.sum(F.col("weather_sensitive").cast("int")).alias("outdoor_count"),
        F.sum((~F.col("weather_sensitive")).cast("int")).alias("indoor_count"),
    )
    .orderBy(F.desc("activity_count"))
)

by_category.show(truncate=False)

+------------+--------------+----------------+------------------+-------------+------------+
|category    |activity_count|avg_duration_hrs|total_duration_hrs|outdoor_count|indoor_count|
+------------+--------------+----------------+------------------+-------------+------------+
|viewpoint   |3             |1.33            |4.0               |3            |0           |
|beach       |3             |2.83            |8.5               |3            |0           |
|cultural    |2             |1.50            |3.0               |0            |2           |
|nature      |2             |3.00            |6.0               |2            |0           |
|hiking      |2             |5.25            |10.5              |2            |0           |
|water_sports|2             |4.50            |9.0               |2            |0           |
|dining      |1             |2.00            |2.0               |0            |1           |
+------------+--------------+----------------+------------------+-----

## 5. Aggregation: weather sensitivity per destination

Groups by destination_id and computes what percentage of activities are weather-sensitive. Useful signal for the agent's rescheduling logic: destinations with more outdoor activities benefit more from weather-aware planning.

In [0]:
by_destination = (
    activities_df
    .groupBy("destination_id")
    .agg(
        F.count("*").alias("total_activities"),
        F.sum(F.col("weather_sensitive").cast("int")).alias("outdoor_activities"),
        F.round(
            100.0 * F.sum(F.col("weather_sensitive").cast("int")) / F.count("*"),
            1,
        ).alias("pct_weather_sensitive"),
        F.round(F.avg("duration_hours"), 2).alias("avg_duration_hrs"),
        F.collect_set("category").alias("categories"),
    )
    .orderBy(F.desc("total_activities"))
)

by_destination.show(truncate=False)

+--------------+----------------+------------------+---------------------+----------------+------------------------------------------------------------------+
|destination_id|total_activities|outdoor_activities|pct_weather_sensitive|avg_duration_hrs|categories                                                        |
+--------------+----------------+------------------+---------------------+----------------+------------------------------------------------------------------+
|1             |15              |12                |80.0                 |2.87            |[dining, cultural, hiking, beach, water_sports, viewpoint, nature]|
+--------------+----------------+------------------+---------------------+----------------+------------------------------------------------------------------+



## 6. Combine into a single analytics DataFrame to persist

Wraps both aggregations into one wide-form analytics table with a `dimension` column marking whether each row is a per-category or per-destination summary.

In [0]:
from pyspark.sql.types import StringType

analytics_by_category = (
    by_category
    .withColumn("dimension", F.lit("category"))
    .withColumn("dimension_value", F.col("category"))
    .withColumn("outdoor_pct",
                F.round(100.0 * F.col("outdoor_count") / F.col("activity_count"), 1))
    .select(
        "dimension",
        "dimension_value",
        F.col("activity_count").alias("total_activities"),
        F.col("outdoor_count").alias("outdoor_activities"),
        F.col("outdoor_pct").alias("pct_weather_sensitive"),
        "avg_duration_hrs",
    )
)

analytics_by_destination = (
    by_destination
    .withColumn("dimension", F.lit("destination"))
    .withColumn("dimension_value", F.col("destination_id").cast(StringType()))
    .select(
        "dimension",
        "dimension_value",
        "total_activities",
        "outdoor_activities",
        "pct_weather_sensitive",
        "avg_duration_hrs",
    )
)

analytics = analytics_by_category.unionByName(analytics_by_destination)

print(f"Combined analytics rows: {analytics.count()}")
analytics.show(truncate=False)

Combined analytics rows: 8
+-----------+---------------+----------------+------------------+---------------------+----------------+
|dimension  |dimension_value|total_activities|outdoor_activities|pct_weather_sensitive|avg_duration_hrs|
+-----------+---------------+----------------+------------------+---------------------+----------------+
|category   |viewpoint      |3               |3                 |100.0                |1.33            |
|category   |beach          |3               |3                 |100.0                |2.83            |
|category   |cultural       |2               |0                 |0.0                  |1.50            |
|category   |nature         |2               |2                 |100.0                |3.00            |
|category   |hiking         |2               |2                 |100.0                |5.25            |
|category   |water_sports   |2               |2                 |100.0                |4.50            |
|category   |dining         

## 7. Write results back to Lakebase

Persists the analytics DataFrame as a new `activity_analytics` table via Spark JDBC. Uses `overwrite` mode so re-running the pipeline replaces prior results.

In [0]:
# Serverless requires individual connection parameters instead of url
port = parsed.port or 5432

(
    analytics.write
    .format("postgresql")
    .option("host", parsed.hostname)
    .option("port", port)
    .option("database", parsed.path.lstrip("/"))
    .option("dbtable", "activity_analytics")
    .option("user", JDBC_USER)
    .option("password", JDBC_PASS)
    .mode("overwrite")
    .save()
)

print("Wrote analytics table back to Lakebase")

Wrote analytics table back to Lakebase


## 8. Verify — read the analytics table back

Reads the freshly-written table via Spark JDBC to confirm the round trip worked. Should show the same rows we wrote.

In [0]:
verify_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url_with_port)
    .option("dbtable", "activity_analytics")
    .option("user", JDBC_USER)
    .option("password", JDBC_PASS)
    .option("driver", "org.postgresql.Driver")
    .load()
)

print(f"Read back {verify_df.count()} analytics rows")
verify_df.orderBy("dimension", F.desc("total_activities")).show(truncate=False)

Read back 8 analytics rows
+-----------+---------------+----------------+------------------+---------------------+----------------+
|dimension  |dimension_value|total_activities|outdoor_activities|pct_weather_sensitive|avg_duration_hrs|
+-----------+---------------+----------------+------------------+---------------------+----------------+
|category   |viewpoint      |3               |3                 |100.0                |1.33            |
|category   |beach          |3               |3                 |100.0                |2.83            |
|category   |hiking         |2               |2                 |100.0                |5.25            |
|category   |water_sports   |2               |2                 |100.0                |4.50            |
|category   |cultural       |2               |0                 |0.0                  |1.50            |
|category   |nature         |2               |2                 |100.0                |3.00            |
|category   |dining         

## Pipeline Summary

This notebook demonstrates a complete Spark data pipeline over the RoamAI activities dataset:

1. **Extract** — read from Lakebase Postgres via Spark JDBC  
2. **Transform** — two aggregations using PySpark DataFrame ops (`groupBy`, `agg`, `avg`, `sum`, `round`, `cast`, `collect_set`, `union`)  
3. **Load** — write back to Lakebase as a new `activity_analytics` table  

In production, this notebook would run on a scheduled Databricks Workflow (daily or weekly) to keep the analytics table fresh as new activities are added. The Streamlit dashboard or agent could then read `activity_analytics` for quick summary stats without re-computing over the full activities table.